# 05 — Geocoder (with reference table)
Matches Address/Location text to Venice coordinates.

Pipeline position: OCR -> Evaluation -> Version Selection -> Statistics -> Geocoder

Inputs: `geospatial/sestiere.geojson`, `geospatial/parishes.geojson`, `geospatial/n_civ.geojson`, `geospatial/venice_location_reference_table.csv` (built once, cached), `clean_pages/page_X/page_X_semantic.csv`


## Imports


In [ ]:
import re
import json
import io
import warnings
from pathlib import Path
from datetime import datetime
import time
import openai

import pandas as pd
import geopandas as gpd
from rapidfuzz import process, fuzz

warnings.filterwarnings("ignore")

from config import (
    CLEAN_PAGES_DIR,
    GEOSPATIAL_DIR,
    DRIVE_ROOT_FOLDER,
    DRIVE_CLEAN_PAGES_FOLDER,
    DRIVE_GEOSPATIAL_FOLDER,
    DRIVE_OUTPUTS_FOLDER,
    OPENAI_KEY_FILE,
    SAVE_MODE,
)

from drive_utils import (
    get_drive_service,
    find_folder,
    get_or_create_folder,
    list_files_in_folder,
    download_text,
    save_or_upload_csv,
    save_or_upload_geojson,
    save_or_upload_text,
    get_run_folder_id,
)


## Configuration
Page selection, LLM repass toggle, excluded pages, local paths.


In [ ]:
PAGES = "all"
# "all"           → geocode all non-excluded pages
# [52, 87, 103]   → specific pages
# range(49, 53)   → contiguous range (49, 50, 51, 52)

# SAVE_MODE is in config.py — imported above.

RUN_LLM_REPASS = True

# Excluded page ranges — always skipped regardless of PAGES setting
SKIP_PAGES      = {1, 2, 6, 7, 8, 9, 161, 490, 571}
INDEX_PAGES     = set(range(10, 36)) | {491, 492, 493}
PROVINCIA_PAGES = set(range(494, 571))
EXCLUDED_PAGES  = SKIP_PAGES | INDEX_PAGES | PROVINCIA_PAGES

# Local paths
CLEAN_PAGES  = Path(CLEAN_PAGES_DIR)
GEO_DIR      = Path(GEOSPATIAL_DIR)
OUTPUTS_DIR  = GEO_DIR / "outputs"

# GeoJSON reference files — always local (committed to repo)
SESTIERE_PATH = GEO_DIR / "sestiere.geojson"
PARISHES_PATH = GEO_DIR / "parishes.geojson"
NCIV_PATH     = GEO_DIR / "n_civ.geojson"
REF_TABLE_PATH = GEO_DIR / "venice_location_reference_table.csv"

SHOW_STATS = True


## Drive helpers
Clean-pages file-ID index (cached, avoids per-file Drive calls).


In [ ]:
_clean_pages_index = None  # {page_num: {filename: file_id}}


def build_clean_pages_index():
    # resolves clean_pages/page_X/ file ids on Drive in one pass, {} if local dirs exist
    global _clean_pages_index
    if _clean_pages_index is not None:
        return _clean_pages_index

    # Check for local page dirs first
    if CLEAN_PAGES.exists():
        local_page_dirs = [
            d for d in CLEAN_PAGES.iterdir()
            if d.is_dir() and re.match(r'^page_\d+$', d.name)
        ]
        if local_page_dirs:
            _clean_pages_index = {}
            return _clean_pages_index

    print("  Building Drive clean_pages index (one-time, ~1-2 min) ...")
    service = get_drive_service()
    root_id = find_folder(service, DRIVE_ROOT_FOLDER)
    if root_id is None:
        _clean_pages_index = {}
        return _clean_pages_index

    cp_folder_id = find_folder(service, DRIVE_CLEAN_PAGES_FOLDER, root_id)
    if cp_folder_id is None:
        _clean_pages_index = {}
        return _clean_pages_index

    subfolders = [
        f for f in list_files_in_folder(service, cp_folder_id)
        if f["mimeType"] == "application/vnd.google-apps.folder"
        and re.match(r'^page_\d+$', f["name"])
    ]

    index = {}
    for folder in subfolders:
        m = re.search(r'(\d+)', folder["name"])
        if not m:
            continue
        page_num = int(m.group(1))
        files = list_files_in_folder(service, folder["id"])
        index[page_num] = {f["name"]: f["id"] for f in files}

    _clean_pages_index = index
    print(f"  Index built: {len(index)} page folders cached.")
    return index


def read_clean_page_csv(page_num):
    # reads clean_pages/page_X/page_X_semantic.csv, local first then Drive index
    filename   = f"page_{page_num}_semantic.csv"
    local_path = CLEAN_PAGES / f"page_{page_num}" / filename
    if local_path.exists():
        return local_path.read_text(encoding="utf-8")

    index = build_clean_pages_index()
    if index:
        page_files = index.get(page_num, {})
        file_id    = page_files.get(filename)
        if file_id is None:
            return None
        return download_text(get_drive_service(), file_id)

    return None


## Run setup
Run tag, output directory, output file paths.


In [1]:

def make_run_tag(pages, excluded):
    # descriptive run tag from the PAGES config
    if pages == "all":
        return "all_pages"
    nums = sorted(set(pages) - excluded)
    if not nums:
        return "empty"
    if len(nums) == 1:
        return f"page_{nums[0]}"
    if nums == list(range(nums[0], nums[-1] + 1)):
        return f"page_{nums[0]}-{nums[-1]}"
    if len(nums) <= 6:
        return "page_" + "_".join(str(n) for n in nums)
    return f"page_{nums[0]}_{nums[-1]}_and_{len(nums)-2}_others"


TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
RUN_TAG   = make_run_tag(PAGES, EXCLUDED_PAGES)
RUN_NAME  = f"{RUN_TAG}_{TIMESTAMP}"
FILE_TAG  = f"ref_table_{RUN_NAME}"

LOCAL_RUN_DIR = OUTPUTS_DIR / RUN_NAME
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV                  = LOCAL_RUN_DIR / f"{FILE_TAG}_geocoded.csv"
OUT_GEOJSON              = LOCAL_RUN_DIR / f"{FILE_TAG}_geocoded.geojson"
OUT_UNMATCHED_SESTIERE   = LOCAL_RUN_DIR / f"{FILE_TAG}_unmatched.csv"
OUT_UNMATCHED_LLM        = LOCAL_RUN_DIR / f"{FILE_TAG}_unmatched_llm_repass.csv"
OUT_UNMATCHED_NOMINATIM  = LOCAL_RUN_DIR / f"{FILE_TAG}_unmatched_nominatim.csv"
OUT_AMBIGUOUS            = LOCAL_RUN_DIR / f"{FILE_TAG}_ambiguous.csv"
OUT_REPORT               = LOCAL_RUN_DIR / f"{FILE_TAG}_report.md"
OUT_LLM_REPASS           = LOCAL_RUN_DIR / f"{FILE_TAG}_llm_repass.csv"
OUT_NOMINATIM_DECISIONS  = LOCAL_RUN_DIR / f"{FILE_TAG}_nominatim_decisions.csv"
OUT_NOMINATIM_STATS      = LOCAL_RUN_DIR / f"{RUN_NAME}_nominatim_stats.md"

print(f"Run        : {RUN_NAME}")
print(f"Output dir : {LOCAL_RUN_DIR}/")



Run        : all_pages_20260721_1901
Output dir : geospatial/outputs/all_pages_20260721_1901/


## Name normalization tables
Sestiere/parish aliases and landmark mappings, from the reference GeoJSONs.


In [ ]:

SESTIERE_CANONICAL = [
    "CANNAREGIO", "SANTA CROCE", "DORSODURO", "SAN POLO",
    "SAN MARCO",  "CASTELLO",    "GIUDECCA",
]

SESTIERE_ALIASES = {
    # San Marco
    "s. marco":     "SAN MARCO",
    "san marco":    "SAN MARCO",
    "s marco":      "SAN MARCO",
    "marco":        "SAN MARCO",
    # Santa Croce
    "s. croce":     "SANTA CROCE",
    "santa croce":  "SANTA CROCE",
    "s croce":      "SANTA CROCE",
    "s.croce":      "SANTA CROCE",
    # Dorsoduro
    "dorsoduro":    "DORSODURO",
    "d. duro":      "DORSODURO",
    "d.duro":       "DORSODURO",

    # Cannaregio
    "cannaregio":   "CANNAREGIO",
    "cann.":        "CANNAREGIO",
    "cannareggio":  "CANNAREGIO",   
    "cann":         "CANNAREGIO",

    # San Polo
    "s. polo":      "SAN POLO",
    "san polo":     "SAN POLO",
    "s polo":       "SAN POLO",
    "polo":         "SAN POLO",
    # Castello
    "castello":     "CASTELLO",
    "cast.":        "CASTELLO",
  
    # Giudecca
    "giudecca":     "GIUDECCA",
    "giudeca":      "GIUDECCA",    
    
    # Sant'Elena
    "sant'elena":   "CASTELLO",
    "s. elena":     "CASTELLO",
    "santa elena":  "CASTELLO",
    "s elena":      "CASTELLO",
    "s.e.":         "CASTELLO",
    "s. elen.":     "CASTELLO",

    "canareggio":   "CANNAREGIO",
    "canrn.":       "CANNAREGIO",
    "giuidecca":    "GIUDECCA",
    "dduro":        "DORSODURO",
}

# implied sestiere from well-known named places that are NOT in our parish GeoJSON but whose sestiere is historically unambiguous.

SESTIERE_IMPLIED = {
    # Giudecca
    "s. eufemia":   "GIUDECCA",
    "sant'eufemia": "GIUDECCA",
    "eufemia":      "GIUDECCA",
    # Santa Croce
    "s. stae":      "SANTA CROCE",
    "san stae":     "SANTA CROCE",
    "stae":         "SANTA CROCE",
    # San Polo
    "s. tomà":      "SAN POLO",
    "san tomà":     "SAN POLO",
    "s. toma":      "SAN POLO",
    "san toma":     "SAN POLO",
    "rialto":       "SAN POLO",
    # Castello
    "s. lio":       "CASTELLO",
    "san lio":      "CASTELLO",
    # Dorsoduro
    "s. vio":       "DORSODURO",
    "san vio":      "DORSODURO",
    "s. trovaso":   "DORSODURO",
    "san trovaso":  "DORSODURO",
    "trovaso":      "DORSODURO",
    "zattere":      "DORSODURO",
    # San Marco
    "s. angelo":    "SAN MARCO",
    "sant'angelo":  "SAN MARCO",
}

PARISH_ALIASES = {
    # San Giovanni Bragora (Castello)
    "bragora":                  "San Giovanni Bragora",
    "s. giovanni bragora":      "San Giovanni Bragora",
    "san giovanni bragora":     "San Giovanni Bragora",
    # Santa Maria Gloriosa dei Frari (San Polo)
    "frari":                    "Santa Maria Gloriosa",
    "s. maria gloriosa":        "Santa Maria Gloriosa",
    "santa maria gloriosa":     "Santa Maria Gloriosa",
    "s. maria dei frari":       "Santa Maria Gloriosa",
    # San Marziale (Cannaregio)
    "s. marziale":              "San Marziale",
    "san marziale":             "San Marziale",
    "marziale":                 "San Marziale",
    # San Geremia (Cannaregio)
    "s. geremia":               "San Geremia",
    "san geremia":              "San Geremia",
    "geremia":                  "San Geremia",
    # San Silvestro (San Polo)
    "s. silvestro":             "San Silvestro",
    "san silvestro":            "San Silvestro",
    "silvestro":                "San Silvestro",
    # San Zaccaria (Castello)
    "s. zaccaria":              "San Zaccaria",
    "san zaccaria":             "San Zaccaria",
    "zaccaria":                 "San Zaccaria",
    # San Salvador (San Marco)
    "s. salvatore":             "San Salvador",
    "san salvador":             "San Salvador",
    "salvador":                 "San Salvador",
    "salvatore":                "San Salvador",
    # San Luca (San Marco)
    "s. luca":                  "San Luca",
    "san luca":                 "San Luca",
    "luca":                     "San Luca",
    # San Cassiano (San Polo)
    "s. cassiano":              "San Cassiano",
    "san cassiano":             "San Cassiano",
    "cassiano":                 "San Cassiano",
    # San Felice (Cannaregio)
    "s. felice":                "San Felice",
    "san felice":               "San Felice",
    "felice":                   "San Felice",
    # San Giacomo dell'Orio (Santa Croce)
    "s. giacomo dall'orio":     "San Giacomo dell'Orio",
    "s. giacomo dell'orio":     "San Giacomo dell'Orio",
    "san giacomo dell'orio":    "San Giacomo dell'Orio",
    "s. giacomo":               "San Giacomo dell'Orio",
    "giacomo dall'orio":        "San Giacomo dell'Orio",
    "giacomo":                  "San Giacomo dell'Orio",
    # San Simeone Profeta (Santa Croce)
    "s. simeone":               "San Simeone Profeta",
    "san simeone":              "San Simeone Profeta",
    "simeone":                  "San Simeone Profeta",
    "s. simeon":                "San Simeone Profeta",
    # San Pantaleone (Dorsoduro)
    "s. pantaleone":            "San Pantaleone",
    "san pantaleone":           "San Pantaleone",
    "pantaleone":               "San Pantaleone",
    "s. pantalon":              "San Pantaleone",
    "pantalon":                 "San Pantaleone",
    # Santo Stefano (San Marco)
    "s. stefano":               "Santo Stefano",
    "santo stefano":            "Santo Stefano",
    "stefano":                  "Santo Stefano",
    # San Canziano (Cannaregio)
    "s. canziano":              "San Canziano",
    "san canziano":             "San Canziano",
    "canziano":                 "San Canziano",
    # Santi Giovanni e Paolo (Castello)
    "s. giovanni e paolo":      "Santi Giovanni e Paolo",
    "ss. giovanni e paolo":     "Santi Giovanni e Paolo",
    "santi giovanni e paolo":   "Santi Giovanni e Paolo",
    "ss. zanipolo":             "Santi Giovanni e Paolo",
    "zanipolo":                 "Santi Giovanni e Paolo",
    # San Pietro (Castello)
    "s. pietro":                "San Pietro",
    "san pietro":               "San Pietro",
    "pietro":                   "San Pietro",
    # Santa Maria Formosa (Castello)
    "s. maria formosa":         "Santa Maria Formosa",
    "santa maria formosa":      "Santa Maria Formosa",
    "formosa":                  "Santa Maria Formosa",
    # Santa Maria Zobenigo (San Marco)
    "s. maria zobenigo":        "Santa Maria Zobenigo",
    "santa maria zobenigo":     "Santa Maria Zobenigo",
    "zobenigo":                 "Santa Maria Zobenigo",
    "s. maria del giglio":      "Santa Maria Zobenigo",
    "giglio":                   "Santa Maria Zobenigo",
    # Santa Maria del Carmine (Dorsoduro)
    "s. maria del carmine":     "Santa Maria del Carmine",
    "santa maria del carmine":  "Santa Maria del Carmine",
    "carmine":                  "Santa Maria del Carmine",
    # Santa Maria del Rosario (Dorsoduro)
    "s. maria del rosario":     "Santa Maria del Rosario",
    "santa maria del rosario":  "Santa Maria del Rosario",
    "rosario":                  "Santa Maria del Rosario",
    "gesuati":                  "Santa Maria del Rosario",
    # San Nicola da Tolentino (Santa Croce)
    "s. nicola":                "San Nicola da Tolentino",
    "san nicola":               "San Nicola da Tolentino",
    "tolentino":                "San Nicola da Tolentino",
    "s. nicola da tolentino":   "San Nicola da Tolentino",
    # Santi Apostoli (Cannaregio)
    "santi apostoli":           "Santi Apostoli",
    "ss. apostoli":             "Santi Apostoli",
    "apostoli":                 "Santi Apostoli",
    # Santi Gervasio e Protasio (Dorsoduro)
    "s. gervasio":              "Santi Gervasio e Protasio",
    "gervasio":                 "Santi Gervasio e Protasio",
    "ss. gervasio":             "Santi Gervasio e Protasio",
    "gervasio e protasio":      "Santi Gervasio e Protasio",
    "s. trovaso":               "Santi Gervasio e Protasio", 
    "san trovaso":              "Santi Gervasio e Protasio",
    "trovaso":                  "Santi Gervasio e Protasio",
    # Santi Ermagora e Fortunato (Cannaregio)
    "ermagora":                 "Santi Ermagora e Fortunato",
    "s. ermagora":              "Santi Ermagora e Fortunato",
    "ss. ermagora":             "Santi Ermagora e Fortunato",
    "ermagora e fortunato":     "Santi Ermagora e Fortunato",
    # San Martino (Castello)
    "s. martino":               "San Martino",
    "san martino":              "San Martino",
    "martino":                  "San Martino",
    # L'Angelo (Dorsoduro/Santa Croce border area)
    "angelo":                   "L'Angelo",
    "l'angelo":                 "L'Angelo",
    # San Marco (parish, San Marco sestiere)
    "par. s. marco":            "San Marco",
    "parrocchia s. marco":      "San Marco",
}


NON_LOCATION_RE = re.compile(
    r"^(p\.\s*(i{1,3}|iv|vi{0,3}|ix|x|t\.?)|piano\s*(i{1,3}|iv|\d)|p\.?\s*t\.?|pt\.)$",
    re.IGNORECASE,
)
CIVIC_NUM_RE    = re.compile(r"\b(\d{1,5})\s*[-–]?\s*([a-zA-Z])?\b")

CIVIC_NUM_DOTLETTER_RE = re.compile(
    r"\b(\d{1,5})\.([a-zA-Z])\.",
    re.IGNORECASE
)


## Reference table
Builds (or loads cached) civic-number -> parish/sestiere/coordinates table.


In [2]:

def build_reference_table():
    print("[1/3] Loading sestiere and parishes ...")
    sestiere      = gpd.read_file(SESTIERE_PATH)
    parishes      = gpd.read_file(PARISHES_PATH)
    parishes_proj = parishes.to_crs(sestiere.crs)

    par_cent = parishes_proj.copy()
    par_cent["geometry"] = parishes_proj.geometry.centroid
    parish_sestiere = (
        gpd.sjoin(
            par_cent[["parish", "geometry"]],
            sestiere[["division_name", "geometry"]],
            how="left", predicate="within",
        )
        .rename(columns={"division_name": "sestiere"})[["parish", "sestiere"]]
    )
    print(f"    → {len(parish_sestiere)} parishes assigned to a sestiere")

    print("[2/3] Loading n_civ ...")
    nciv   = gpd.read_file(NCIV_PATH, engine="pyogrio", on_invalid="ignore")
    before = len(nciv)
    nciv   = nciv[~nciv.geometry.isna()]
    nciv   = nciv[nciv.geometry.is_valid]
    nciv   = nciv[~nciv.geometry.is_empty]
    nciv   = nciv[nciv.geometry.length > 0]
    dropped = before - len(nciv)
    if dropped:
        print(f"    → Dropped {dropped:,} bad geometries")

    nciv_proj            = nciv.to_crs(sestiere.crs)
    nciv_proj            = nciv_proj[["NC_NUM", "NC_LETT", "geometry"]].copy()
    nciv_proj["geometry"] = nciv_proj.geometry.centroid
    nciv_proj["NC_LETT"]  = nciv_proj["NC_LETT"].replace("_", "").fillna("").str.strip()
    print(f"    → {len(nciv_proj):,} door entries loaded")

    print("[3/3] Spatial join: door centroids → parishes ...")
    nciv_joined = gpd.sjoin(
        nciv_proj,
        parishes_proj[["parish", "geometry"]],
        how="left", predicate="within",
    )
    nciv_joined = nciv_joined.merge(parish_sestiere, on="parish", how="left")
    nciv_wgs84  = nciv_joined.to_crs("EPSG:4326")
    nciv_wgs84["lon"] = nciv_wgs84.geometry.x
    nciv_wgs84["lat"] = nciv_wgs84.geometry.y

    ref = nciv_wgs84[
        ["NC_NUM", "NC_LETT", "parish", "sestiere", "lon", "lat"]
    ].dropna(subset=["parish", "sestiere"]).copy()
    ref["NC_NUM"] = ref["NC_NUM"].astype(int) 

    unmatched = nciv_joined["parish"].isna().sum()
    print(f"    → Reference table: {len(ref):,} entries | "
          f"outside all parishes: {unmatched:,}")
    ref["parish"] = ref["parish"].str.strip().str.replace("_", " ", regex=False)
    return ref


if REF_TABLE_PATH.exists():
    print(f"Loading cached reference table from {REF_TABLE_PATH} ...")
    ref = pd.read_csv(REF_TABLE_PATH, dtype={"NC_LETT": str})
    ref["NC_NUM"]   = ref["NC_NUM"].astype(int)
    ref["NC_LETT"]  = ref["NC_LETT"].fillna("").str.strip()
    ref["parish"]   = ref["parish"].str.strip().str.replace("_", " ", regex=False)
    ref["sestiere"] = ref["sestiere"].str.strip()
    print(f"    → {len(ref):,} entries loaded")
else:
    print("No cached reference table — building from GeoJSON files ...")
    ref = build_reference_table()
    ref.to_csv(REF_TABLE_PATH, index=False)
    print(f"    → Saved to {REF_TABLE_PATH}")

parish_list = ref["parish"].dropna().unique().tolist()



Loading cached reference table from geospatial/venice_location_reference_table.csv ...
    → 35,934 entries loaded



## Address parsing
Parses raw Address/Location text into civic number, letter, sestiere, parish.


In [ ]:

STREET_PREFIX_RE = re.compile(
    r"\b(via|viale|strada|sinistra|destra|calle)\b",
    re.IGNORECASE
)

OUTSIDE_VENICE_RE = re.compile(
    r"\b("
    r"burano|murano|mestre|marghera|porto marghera|lido|"
    r"chioggia|zelarino|favaro veneto|favaro|pellestrina|"
    r"dese|carpenedo|jesolo|spinea|mira|dolo|chirignago|"
    r"tessera|martellago|mirano|salzano|mogliano|"
    r"cavallino|treporti|malamocco|"
    r"s\.\s*pietro in volta|"
    r"annone veneto|campagnalupia|campolongo maggiore|"
    r"camponogara|caorle|cavarzere|ceggia|"
    r"cinto caomaggiore|cona|concordia sagittaria|"
    r"fiess[eo]\s+d.artico|"
    r"fossalta di piave|fossalta di portogruaro|"
    r"foss[oò]'?|grisolera|gruaro|marcon|meolo|"
    r"musile di piave|noale|noventa di piave|pianiga|"
    r"portogruaro|pramaggiore|"
    r"s\.\s*dona'?\s+di\s+piave|san\s+don[aà]\s+di\s+piave|"
    r"s\.\s*maria\s+di\s+sala|santa\s+maria\s+di\s+sala|"
    r"s\.\s*michele\s+al\s+tagliamento|san\s+michele\s+al\s+tagliamento|"
    r"s\.\s*michele\s+del\s+quarto|"
    r"s\.\s*stino\s+di\s+livenza|san\s+stino\s+di\s+livenza|"
    r"teglio veneto|torre di mosto|vigonovo|"
    r"scorz[eè]|"
    r"viale delle industrie"
    r")\b",
    re.IGNORECASE
)

_ALIAS_RE_CACHE = {}


def _word_in(alias, t):
    # word-boundary alias check (plain substring let 'polo'/'marco' match inside 'Popolo'/'Marcolin'); t is apostrophe-stripped same as parse_address._safe so sant'elena etc still match (2026-08-21)
    rx = _ALIAS_RE_CACHE.get(alias)
    if rx is None:
        alias_norm = re.sub(r"[’'`]", "", alias)
        rx = re.compile(r"\b" + re.escape(alias_norm) + r"\b")
        _ALIAS_RE_CACHE[alias] = rx
    return rx.search(t) is not None


def _match_parish(text, parish_list):
    t = text.lower().strip()
    if "in volta" in t:
        # "S. Pietro in Volta" (Pellestrina) is not the Castello "San Pietro" parish
        return None
    for alias, canonical in PARISH_ALIASES.items():
        if _word_in(alias, t):
            return canonical
    if len(t) >= 12:
        hit = process.extractOne(
            t, [p.lower() for p in parish_list],
            scorer=fuzz.token_sort_ratio, score_cutoff=88,
        )
        if hit:
            return parish_list[[p.lower() for p in parish_list].index(hit[0])]
    return None


def _has_location_context(text):
    t = text.lower()
    return (
        any(_word_in(alias, t) for alias in SESTIERE_ALIASES) or
        any(_word_in(alias, t) for alias in PARISH_ALIASES) or
        any(_word_in(alias, t) for alias in SESTIERE_IMPLIED)
    )

def parse_address(row, parish_list):
    out = dict(
        civic_num=None,
        civic_lett=None,
        sestiere=None,
        parish=None,
        sestiere_implied=False,   # True when sestiere came from SESTIERE_IMPLIED
    )

    def _safe(val):
        s = str(val).strip() if pd.notna(val) else ""
        s = "" if s.lower() in ("nan", "none") else s
        # strip stray apostrophes/backticks that OCR sometimes inserts mid-word (e.g. "S. Mar'co")
        return re.sub(r"[’'`]", "", s)

    addr = _safe(row.get("Address", ""))
    loc  = _safe(row.get("Location", ""))

    # Step 1: extract civic number — Address first, then Location.
    # Try dot-letter pattern first (e.g. "4392.B."), then standard pattern.
    # Require location context in EITHER field.
    for text in [addr, loc]:
        if out["civic_num"] is not None:
            break
        if not (_has_location_context(addr) or _has_location_context(loc)):
            break  # no location context in either field — skip both
        # Try dot-letter pattern first: "4392.B."
        for m in CIVIC_NUM_DOTLETTER_RE.finditer(text):
            num  = int(m.group(1))
            lett = m.group(2).upper()
            out["civic_num"]  = num
            out["civic_lett"] = lett
            break
        if out["civic_num"] is not None:
            break
        # Standard pattern: "4392", "4392-B", "4392 B"
        for m in CIVIC_NUM_RE.finditer(text):
            num  = int(m.group(1))
            lett = (m.group(2) or "").upper()
            out["civic_num"]  = num
            out["civic_lett"] = lett
            break

    # Step 2: extract sestiere and parish — Address first, then Location.
    # For sestiere, we track whether it came from a direct alias or an implied landmark alias (SESTIERE_IMPLIED), since these have different reliability levels.
    for text in [addr, loc]:
        if not text or NON_LOCATION_RE.match(text.strip()):
            continue
        if out["parish"] is None:
            out["parish"] = _match_parish(text, parish_list)
        if out["sestiere"] is None:
            t = text.lower().strip()
            # Priority 1: direct sestiere alias
            for alias, canonical in SESTIERE_ALIASES.items():
                if _word_in(alias, t):
                    out["sestiere"] = canonical
                    out["sestiere_implied"] = False
                    break
            # Priority 2: implied sestiere from well-known landmarks
            if out["sestiere"] is None:
                for alias, canonical in SESTIERE_IMPLIED.items():
                    if _word_in(alias, t):
                        out["sestiere"] = canonical
                        out["sestiere_implied"] = True
                        break
            # Priority 3: fuzzy match on canonical sestiere names
            if out["sestiere"] is None and len(t) >= 10:
                hit = process.extractOne(
                    t, [s.lower() for s in SESTIERE_CANONICAL],
                    scorer=fuzz.token_sort_ratio, score_cutoff=85,
                )
                if hit:
                    out["sestiere"] = SESTIERE_CANONICAL[
                        [s.lower() for s in SESTIERE_CANONICAL].index(hit[0])
                    ]
                    out["sestiere_implied"] = False
        if out["parish"] and out["sestiere"]:
            break

    return out



## Geocoding function
Matches a parsed address against the reference table.


In [ ]:
def geocode_row(parsed, ref):
    FAIL = dict(
        matched_lat=None, matched_lon=None,
        matched_parish=None, matched_sestiere=None,
        matched_lett=None, all_doors=None,
        match_confidence="no_match", match_note="",
    )

    def _clean(val):
        # stripped string, or None for NaN/None/empty
        if val is None:
            return None
        s = str(val).strip()
        return None if s.lower() in ("", "nan", "none") else s

    num      = parsed["civic_num"]
    num      = None if (num is None or (isinstance(num, float) and pd.isna(num))) else int(num)
    lett     = _clean(parsed["civic_lett"]) or ""
    sestiere = _clean(parsed["sestiere"])
    parish   = _clean(parsed["parish"])

    # Determine sestiere source — track whether it came from a direct sestiere alias, a landmark inference, or will be derived from parish
    sestiere_implied = parsed.get("sestiere_implied", False)
    if isinstance(sestiere_implied, float):   # NaN safety from parsed_df
        sestiere_implied = False
    sestiere_source = "implied_from_landmark" if sestiere_implied else "direct"

    # Step 1: no civic number → immediate fail
    if num is None:
        return {**FAIL, "match_note": "no_civic_number"}

    # Step 2: if sestiere missing, try to derive from parish.
    if sestiere is None and parish is not None:
        lookup = ref.loc[
            ref["parish"].str.strip().str.lower() == parish.strip().lower(),
            "sestiere"
        ].dropna()
        if not lookup.empty:
            sestiere = lookup.iloc[0]
            sestiere_source = "derived_from_parish"

    # Step 3: still no sestiere → fail
    if sestiere is None:
        return {**FAIL, "match_note": "no_sestiere_or_parish"}

    # Step 4: look up all doors matching (NC_NUM, sestiere).
    # Civic numbers are unique within a sestiere, so this returns one physical location — possibly with multiple lettered door variants.
    cands = ref[
        (ref["NC_NUM"] == num) &
        (ref["sestiere"].str.upper() == sestiere.upper())
    ].copy()

    if cands.empty:
        return {**FAIL, "match_note": f"num={num} not found in sestiere={sestiere}"}

    def _doors_json(subset):
        return json.dumps([
            {"lett": str(r["NC_LETT"]), "lat": round(r["lat"], 6),
             "lon": round(r["lon"], 6)}
            for _, r in subset.iterrows()
        ])

    # Step 5: only one door at this number → exact match
    if len(cands) == 1:
        r = cands.iloc[0]
        return dict(
            matched_lat=r["lat"], matched_lon=r["lon"],
            matched_parish=r["parish"], matched_sestiere=r["sestiere"],
            matched_lett=r["NC_LETT"], all_doors=None,
            match_confidence=f"exact_sestiere_{sestiere_source}",
            match_note=f"num={num}, sestiere={sestiere} ({sestiere_source}), single door"
        )

    # Step 6: multiple doors (letter variants) — try to match the letter extracted from the almanac row. Also try common OCR letter/digit confusions if exact match fails.
    if lett:
        # Try exact match first
        with_lett = cands[cands["NC_LETT"].str.upper() == lett.upper()]
        if len(with_lett) == 1:
            r = with_lett.iloc[0]
            return dict(
                matched_lat=r["lat"], matched_lon=r["lon"],
                matched_parish=r["parish"], matched_sestiere=r["sestiere"],
                matched_lett=r["NC_LETT"], all_doors=None,
                match_confidence=f"exact_sestiere_with_letter_{sestiere_source}",
                match_note=f"num={num}{lett}, sestiere={sestiere} ({sestiere_source}), matched letter"
            )

        # Exact match failed — try correcting common OCR digit/letter confusions
        # 0↔O, 1↔I, 8↔B, 6↔G
        OCR_LETTER_FIXES = {"0": "O", "1": "I", "8": "B", "6": "G"}
        corrected_lett = OCR_LETTER_FIXES.get(lett.upper(), lett.upper())
        if corrected_lett != lett.upper():
            with_lett_fixed = cands[cands["NC_LETT"].str.upper() == corrected_lett]
            if len(with_lett_fixed) == 1:
                r = with_lett_fixed.iloc[0]
                return dict(
                    matched_lat=r["lat"], matched_lon=r["lon"],
                    matched_parish=r["parish"], matched_sestiere=r["sestiere"],
                    matched_lett=r["NC_LETT"], all_doors=None,
                    match_confidence=f"exact_sestiere_with_letter_ocr_corrected_{sestiere_source}",
                    match_note=f"num={num}, letter {lett}→{corrected_lett} (OCR digit/letter correction), "
                               f"sestiere={sestiere} ({sestiere_source})"
                )

    # Step 7: no letter in almanac row, or letter not found among variants.
    # Prefer the unlettered row (blank NC_LETT = main entrance) if it exists.
    unlettered = cands[cands["NC_LETT"] == ""]
    if len(unlettered) == 1:
        r = unlettered.iloc[0]
        return dict(
            matched_lat=r["lat"], matched_lon=r["lon"],
            matched_parish=r["parish"], matched_sestiere=r["sestiere"],
            matched_lett=r["NC_LETT"], all_doors=_doors_json(cands),
            match_confidence=f"exact_sestiere_main_entrance_{sestiere_source}",
            match_note=f"num={num}, sestiere={sestiere} ({sestiere_source}), "
                       f"used unlettered row; {len(cands)} letter variants exist"
        )

    # Step 8: no unlettered row, no matching letter → use first variant, flag as approximate. Coordinates are still correct at building level.
    r = cands.iloc[0]
    return dict(
        matched_lat=r["lat"], matched_lon=r["lon"],
        matched_parish=r["parish"], matched_sestiere=r["sestiere"],
        matched_lett=r["NC_LETT"], all_doors=_doors_json(cands),
        match_confidence=f"approximate_sestiere_first_variant_{sestiere_source}",
        match_note=f"num={num}, sestiere={sestiere} ({sestiere_source}), "
                   f"no letter match; used first of {len(cands)} variants"
    )


## Determine pages to process


In [3]:

def resolve_pages(pages_config, excluded):
    # sorted list of page numbers to process
    if pages_config == "all":
        # Discover from Drive index or local dirs
        index = build_clean_pages_index()
        if index:
            all_nums = sorted(index.keys())
        else:
            all_nums = sorted(
                int(re.search(r'(\d+)', d.name).group(1))
                for d in CLEAN_PAGES.iterdir()
                if d.is_dir() and re.match(r'^page_\d+$', d.name)
            )
        return [n for n in all_nums if n not in excluded]

    if isinstance(pages_config, range):
        return [n for n in pages_config if n not in excluded]

    if isinstance(pages_config, list):
        return [n for n in sorted(pages_config) if n not in excluded]

    raise ValueError(f"PAGES must be 'all', a list, or a range. Got: {pages_config!r}")


page_nums = resolve_pages(PAGES, EXCLUDED_PAGES)
print(f"\nPages to geocode: {len(page_nums)}")
if len(page_nums) <= 20:
    print(f"  {page_nums}")
else:
    print(f"  First 5: {page_nums[:5]} ... Last 5: {page_nums[-5:]}")




Pages to geocode: 457
  First 5: [3, 4, 5, 36, 37] ... Last 5: [489, 572, 573, 574, 575]



## Load all CSVs
Bulk-loads every page's semantic CSV into one DataFrame.


In [4]:

print(f"\nLoading {len(page_nums)} CSVs ...")
frames = []
missing_pages = []

for pn in page_nums:
    text = read_clean_page_csv(pn)
    if text is None:
        missing_pages.append(pn)
        continue
    try:
        page_df = pd.read_csv(io.StringIO(text), dtype=str)
        page_df.columns = [str(c).strip() for c in page_df.columns]
        page_df = page_df.loc[:, ~page_df.columns.duplicated(keep="first")]

        # Merge Journal/Journals into one column
        if "Journal" in page_df.columns and "Journals" in page_df.columns:
            page_df["Journal"] = page_df["Journal"].fillna(
                page_df["Journals"]
            )
            page_df = page_df.drop(columns=["Journals"])
        elif "Journals" in page_df.columns:
            page_df = page_df.rename(columns={"Journals": "Journal"})
            
        page_df["_page_num"]    = pn
        page_df["_source_file"] = f"page_{pn}_semantic.csv"
        frames.append(page_df)
    except Exception as e:
        print(f"  WARNING: could not parse page_{pn}: {e}")
        missing_pages.append(pn)

if missing_pages:
    print(f"  Could not load: {missing_pages}")

if not frames:
    raise ValueError("No CSVs could be loaded — check PAGES config and Drive connection.")

# All page numbers the index knows about
all_known_pages = set(build_clean_pages_index().keys()) if build_clean_pages_index() else set()

# Cleaner version using the frames we actually got
loaded_page_nums = set()
for frame in frames:
    if "_page_num" in frame.columns:
        loaded_page_nums.update(frame["_page_num"].unique())

not_loaded = [pn for pn in page_nums if pn not in loaded_page_nums]
print(f"\nPages requested but not loaded ({len(not_loaded)}): {not_loaded}")

# Also show pages in index but excluded
in_index_not_requested = sorted(
    (set(all_known_pages) - set(page_nums)) - EXCLUDED_PAGES
)
if in_index_not_requested:
    print(f"Pages in Drive index but not requested (excluded): "
          f"{len(in_index_not_requested)} pages")
    

# Retry pages that failed to load — transient Drive errors
if missing_pages:
    print(f"  Retrying {len(missing_pages)} failed pages ...")
    still_missing = []
    for pn in missing_pages:
        text = read_clean_page_csv(pn)
        if text is None:
            still_missing.append(pn)
            continue
        try:
            page_df = pd.read_csv(io.StringIO(text), dtype=str)
            page_df.columns = [str(c).strip() for c in page_df.columns]
            page_df = page_df.loc[:, ~page_df.columns.duplicated(keep="first")]
            page_df["_page_num"]    = pn
            page_df["_source_file"] = f"page_{pn}_semantic.csv"
            frames.append(page_df)
            print(f"    Retry succeeded: page_{pn}")
        except Exception as e:
            still_missing.append(pn)
            print(f"    Retry failed: page_{pn}: {e}")
    missing_pages = still_missing

df = pd.concat(frames, ignore_index=True)
print(f"  Loaded {len(frames)} pages → {len(df)} total rows")


Loading 457 CSVs ...

Pages requested but not loaded (0): []
  Loaded 457 pages → 24626 total rows



## Parse addresses (run)


In [5]:

print("\nParsing addresses ...")
parsed_list = df.apply(lambda r: parse_address(r, parish_list), axis=1).tolist()
parsed_df = pd.DataFrame(parsed_list)
df["civic_num"]        = parsed_df["civic_num"]
df["civic_lett"]       = parsed_df["civic_lett"]
df["sestiere_parsed"]  = parsed_df["sestiere"]
df["parish_parsed"]    = parsed_df["parish"]
df["sestiere_implied"] = parsed_df["sestiere_implied"].fillna(False)
print(f"  Rows with civic number        : {df['civic_num'].notna().sum()}")
print(f"  Rows with sestiere (direct)   : {df[df['sestiere_parsed'].notna() & ~df['sestiere_implied']].shape[0]}")
print(f"  Rows with sestiere (implied)  : {int(df['sestiere_implied'].sum())}")
print(f"  Rows with parish only         : {(df['parish_parsed'].notna() & df['sestiere_parsed'].isna()).sum()}")
print(f"  Rows with both                : {(df['parish_parsed'].notna() & df['sestiere_parsed'].notna()).sum()}")
print(f"  Rows with neither             : {(df['parish_parsed'].isna() & df['sestiere_parsed'].isna()).sum()}")


Parsing addresses ...
  Rows with civic number        : 15956
  Rows with sestiere (direct)   : 14666
  Rows with sestiere (implied)  : 1000
  Rows with parish only         : 1507
  Rows with both                : 704
  Rows with neither             : 7453



## Geocode (run)


In [6]:

print("\nGeocoding ...")
geo_list = parsed_df.apply(lambda r: geocode_row(r, ref), axis=1).tolist()
geo_df   = pd.DataFrame(geo_list)
for col in geo_df.columns:
    df[col] = geo_df[col]

matched_before = df["matched_lat"].notna().sum()
print(f"  Matched before propagation: {matched_before} / {len(df)}")


Geocoding ...
  Matched before propagation: 14204 / 24626



## Propagation
Fills matches across rows sharing an address, within and across pages.


In [ ]:

# Pass 1: within-page propagation
print("\nPropagating matches within pages ...")
within_propagated = 0
for page_num, page_group in df.groupby("_page_num"):
    matched_in_page = page_group[page_group["matched_lat"].notna()]
    addr_map = {}
    for _, row in matched_in_page.iterrows():
        key = str(row.get("Address", "") or "").strip().lower()
        if key and key not in addr_map:
            addr_map[key] = row

    for idx, row in page_group[page_group["matched_lat"].isna()].iterrows():
        key = str(row.get("Address", "") or "").strip().lower()
        if key and key in addr_map:
            src = addr_map[key]
            df.at[idx, "matched_lat"]      = src["matched_lat"]
            df.at[idx, "matched_lon"]      = src["matched_lon"]
            df.at[idx, "matched_parish"]   = src["matched_parish"]
            df.at[idx, "matched_sestiere"] = src["matched_sestiere"]
            df.at[idx, "matched_lett"]     = src.get("matched_lett")
            df.at[idx, "all_doors"]        = src.get("all_doors")
            df.at[idx, "match_confidence"] = "propagated_from_same_address"
            df.at[idx, "match_note"]       = (
                f"within-page propagation from "
                f"{src['match_confidence']}"
            )
            within_propagated += 1

print(f"  Within-page propagated: {within_propagated}")


# Pass 2: cross-page propagation

print("Propagating matches across pages (sestiere-gated) ...")
cross_map = {}
for _, row in df[df["matched_lat"].notna()].iterrows():
    addr_key = str(row.get("Address", "") or "").strip().lower()
    sest     = str(row.get("matched_sestiere", "") or "").strip().upper()
    key      = (addr_key, sest)
    if addr_key and sest and key not in cross_map:
        cross_map[key] = row

cross_propagated = 0
for idx, row in df[df["matched_lat"].isna()].iterrows():
    addr_key = str(row.get("Address", "") or "").strip().lower()
    sest     = str(row.get("sestiere_parsed", "") or "").strip().upper()
    key      = (addr_key, sest)
    if addr_key and sest and key in cross_map:
        src = cross_map[key]
        df.at[idx, "matched_lat"]      = src["matched_lat"]
        df.at[idx, "matched_lon"]      = src["matched_lon"]
        df.at[idx, "matched_parish"]   = src["matched_parish"]
        df.at[idx, "matched_sestiere"] = src["matched_sestiere"]
        df.at[idx, "matched_lett"]     = src.get("matched_lett")
        df.at[idx, "all_doors"]        = src.get("all_doors")
        df.at[idx, "match_confidence"] = "propagated_cross_page"
        df.at[idx, "match_note"]       = (
            f"cross-page propagation from page_{src['_page_num']}; "
            f"original: {src['match_confidence']}"
        )
        cross_propagated += 1

print(f"  Cross-page propagated: {cross_propagated}")
print(f"  Total matched: {df['matched_lat'].notna().sum()} / {len(df)}")


#####################DEBUG

print("\n--- Propagation diagnostics ---")
# Why was within-page propagation so low?
addr_non_empty = df["Address"].notna() & (df["Address"].str.strip() != "")
print(f"  Rows with non-empty Address field: {addr_non_empty.sum()}")
print(f"  Unique non-empty Address strings : {df.loc[addr_non_empty, 'Address'].str.strip().str.lower().nunique()}")
print(f"  Matched rows with non-empty Address: {(df['matched_lat'].notna() & addr_non_empty).sum()}")

# Show sample of address strings to understand format
print("\n  Sample Address values (first 10 non-empty):")
print(df.loc[addr_non_empty, "Address"].head(10).tolist())

# Cross-page: check why cross_map was empty or unmatched
print(f"\n  Cross-map size (unique addr+sestiere combos): {len(cross_map)}")
print(f"  Unmatched rows with non-empty Address + sestiere_parsed:")
cross_candidates = df[
    df["matched_lat"].isna() &
    df["Address"].notna() &
    (df["Address"].str.strip() != "") &
    df["sestiere_parsed"].notna()
]
print(f"    {len(cross_candidates)} rows could potentially receive cross-page propagation")
##############################


# ── POST-PROPAGATION CLEANUP ──────────────────────────────
# Flag matched rows where Address or Location contains an explicit outside-Venice place name. These may be wrong propagations and should be reviewed. Tier 4


suspicious_mask = (
    df["matched_lat"].notna() &
    (
        df["Address"].fillna("").str.contains(OUTSIDE_VENICE_RE, na=False) |
        df["Location"].fillna("").str.contains(OUTSIDE_VENICE_RE, na=False)
    )
)

df.loc[suspicious_mask, "match_note"] = (
    df.loc[suspicious_mask, "match_note"] +
    " | WARNING: outside-Venice place name in Address/Location"
)

print(f"  Matched rows with outside-Venice warning: {suspicious_mask.sum()}")
print("\n  Sample suspicious rows:")
print(df[suspicious_mask][[
    "Name", "Address", "Location", "_page_num",
    "match_confidence", "matched_sestiere"
]].head(10).to_string())


Propagating matches within pages ...
  Within-page propagated: 91
Propagating matches across pages (sestiere-gated) ...
  Cross-page propagated: 0
  Total matched: 14295 / 24626

--- Propagation diagnostics ---
  Rows with non-empty Address field: 24514
  Unique non-empty Address strings : 15268
  Matched rows with non-empty Address: 14295

  Sample Address values (first 10 non-empty):
['Venezia - Cannaregio N. 5630', 'Cannaregio N. 5659', 'S. Luca N. 4511', 'Lido - Viale S. M. Elisabetta N. 31', 'S. Luca N. 4480', 'CAMPO S. BARTOLOMEO N. 5379', 'Laboratorio : CANNAREGIO 1956, Negozi : S. MARCO, MERCERIE DEL CAPITELLO 4581, SOTTOPORT. delle ACQUE 5013', 'Arciprete della Patriarcale Basilica Lateranese Vicario Generale di S. S.', 'Arciprete della Patriarcale Basilica Lateranese Vicario Generale di S. S.', 'Arciprete della Patriarcale Basilica Lateranese Vicario Generale di S. S.']

  Cross-map size (unique addr+sestiere combos): 8978
  Unmatched rows with non-empty Address + sestiere_pa

## Build report


In [8]:

DOCUMENT_SECTIONS = {
    "cover_ads":       list(range(1,   14)),
    "index":           list(range(14,  36)),
    "religious":       list(range(36,  38)) + list(range(86, 98)),
    "government":      list(range(38,  86)),
    "professionisti":  list(range(108, 129)),
    "industria":       list(range(132, 319)),
    "indice_generale": list(range(332, 488)),
    "provincia":       list(range(494, 571)),
}


def get_section(page_num):
    for name, pages in DOCUMENT_SECTIONS.items():
        if page_num in pages:
            return name
    return "other"


MATCH_CONFIDENCES = {
    "exact_sestiere_direct",
    "exact_sestiere_derived_from_parish",
    "exact_sestiere_implied_from_landmark",
    "exact_sestiere_with_letter_direct",
    "exact_sestiere_with_letter_derived_from_parish",
    "exact_sestiere_with_letter_implied_from_landmark",
    "exact_sestiere_main_entrance_direct",
    "exact_sestiere_main_entrance_derived_from_parish",
    "exact_sestiere_main_entrance_implied_from_landmark",
    "approximate_sestiere_first_variant_direct",
    "approximate_sestiere_first_variant_derived_from_parish",
    "approximate_sestiere_first_variant_implied_from_landmark",
    "propagated_from_same_address",
    "propagated_cross_page",
"exact_sestiere_with_letter_ocr_corrected_direct",
"exact_sestiere_with_letter_ocr_corrected_derived_from_parish",
"exact_sestiere_with_letter_ocr_corrected_implied_from_landmark",
}

df["_section"] = df["_page_num"].apply(get_section)


# ── GEOCODING TIER ───────────────────────────────────────
# Tier 1: exact match, direct sestiere — most reliable
# Tier 2: exact match, parish-derived or implied sestiere
# Tier 3: approximate match (wrong/missing letter) or propagated
# Tier 4: matched but has outside-Venice warning — needs review
# Tier 0: unmatched

TIER_MAP = {
    "exact_sestiere_direct":                                    1,
    "exact_sestiere_with_letter_direct":                        1,
    "exact_sestiere_derived_from_parish":                       2,
    "exact_sestiere_with_letter_derived_from_parish":           2,
    "exact_sestiere_implied_from_landmark":                     2,
    "exact_sestiere_with_letter_implied_from_landmark":         2,
    "exact_sestiere_main_entrance_direct":                      2,
    "exact_sestiere_main_entrance_derived_from_parish":         2,
    "exact_sestiere_main_entrance_implied_from_landmark":       2,
    "exact_sestiere_with_letter_ocr_corrected_direct":                  2,
"exact_sestiere_with_letter_ocr_corrected_derived_from_parish":     2,
"exact_sestiere_with_letter_ocr_corrected_implied_from_landmark":   2,
    "approximate_sestiere_first_variant_direct":                3,
    "approximate_sestiere_first_variant_derived_from_parish":   3,
    "approximate_sestiere_first_variant_implied_from_landmark": 3,
    "propagated_from_same_address":                             3,
    "propagated_cross_page":                                    3,

}

df["geocoding_tier"] = df["match_confidence"].map(TIER_MAP).fillna(0).astype(int)

# Upgrade tier 1/2/3 to tier 4 if outside-Venice warning present
outside_warn_mask = df["match_note"].fillna("").str.contains("WARNING: outside-Venice", na=False)
df.loc[outside_warn_mask & (df["geocoding_tier"] > 0), "geocoding_tier"] = 4

print(f"\n  Geocoding tier distribution:")
for tier, label in {
    0: "unmatched",
    1: "exact, direct sestiere",
    2: "exact, parish/implied sestiere",
    3: "approximate or propagated",
    4: "matched but needs review (outside-Venice warning)",
}.items():
    count = (df["geocoding_tier"] == tier).sum()
    print(f"    Tier {tier} ({label}): {count}")


def _page_stats(grp):
    total   = len(grp)
    matched = grp["match_confidence"].isin(MATCH_CONFIDENCES).sum()
    unm     = grp[~grp["match_confidence"].isin(MATCH_CONFIDENCES)]
    return {
        "total":       total,
        "matched":     matched,
        "pct":         round(100 * matched / total, 1) if total else 0,
        "no_num":      unm["civic_num"].isna().sum(),
        "not_in_ref":  unm["match_note"].str.contains(
                           "not found in sestiere", na=False).sum(),
        "no_sestiere": (
    unm["sestiere_parsed"].isna() &
    unm["parish_parsed"].isna()
).sum(),
    }


section_stats = {}
for section, grp in df.groupby("_section"):
    section_stats[section] = _page_stats(grp)

page_stats_rows = []
for page_num, grp in df.groupby("_page_num"):
    s = _page_stats(grp)
    s["page_num"] = page_num
    s["section"]  = get_section(page_num)
    page_stats_rows.append(s)
page_stats_df = pd.DataFrame(page_stats_rows).sort_values("page_num")

total_rows    = len(df)
total_matched = df["match_confidence"].isin(MATCH_CONFIDENCES).sum()
conf_counts   = df["match_confidence"].value_counts()


def build_report():
    lines = [
        "# Venice Almanac — Geocoding Report",
        f"Run: `{RUN_NAME}`",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        "",
        "---",
        "",
        "## Overall Summary",
        "",
        f"| Metric | Value |",
        f"|--------|-------|",
        f"| Pages processed | {len(page_nums)} |",
        f"| Total rows | {total_rows} |",
        f"| Matched (all confidence levels) | {total_matched} ({100*total_matched/max(total_rows,1):.1f}%) |",
        f"| Within-page propagated | {within_propagated} |",
        f"| Cross-page propagated | {cross_propagated} |",
        f"| Unmatched | {total_rows - total_matched} |",
        "",
        "## Match Confidence Breakdown",
        "",
        "| Confidence | Count | % |",
        "|------------|-------|---|",
    ]
    for conf, count in conf_counts.items():
        lines.append(f"| {conf} | {count} | {100*count/max(total_rows,1):.1f}% |")

    lines += [
        "",
        "## Results by Section",
        "",
        "| Section | Total | Matched | % | No num | Not in ref | No sestiere |",
        "|---------|-------|---------|---|--------|------------|-------------|",
    ]
    for section, s in sorted(section_stats.items()):
        lines.append(
            f"| {section} | {s['total']} | {s['matched']} | {s['pct']}% | "
            f"{s['no_num']} | {s['not_in_ref']} | {s['no_sestiere']} |"
        )

    lines += [
        "",
        "## Results by Page (top 30 by match rate)",
        "",
        "| Page | Section | Total | Matched | % |",
        "|------|---------|-------|---------|---|",
    ]
    for _, row in page_stats_df.nlargest(30, "pct").iterrows():
        lines.append(
            f"| {int(row['page_num'])} | {row['section']} | "
            f"{row['total']} | {row['matched']} | {row['pct']}% |"
        )

    lines += [
        "",
        "## Pages with Lowest Match Rate (bottom 20)",
        "",
        "| Page | Section | Total | Matched | % | No num | Not in ref | No sestiere |",
        "|------|---------|-------|---------|---|--------|------------|-------------|",
    ]
    for _, row in page_stats_df.nsmallest(20, "pct").iterrows():
        lines.append(
            f"| {int(row['page_num'])} | {row['section']} | "
            f"{row['total']} | {row['matched']} | {row['pct']}% | "
            f"{row['no_num']} | {row['not_in_ref']} | {row['no_sestiere']} |"
        )

    lines += ["", "---", "", "_End of report_"]
    return "\n".join(lines)


report_text = build_report()
print("\n" + "=" * 55)
print(report_text[:3000])  # preview first part in notebook
if len(report_text) > 3000:
    print(f"\n... (truncated — full report saved to {OUT_REPORT.name})")
print("=" * 55)


# Venice Almanac — Geocoding Report
Run: `all_pages_20260721_1901`
Generated: 2026-07-21 19:03

---

## Overall Summary

| Metric | Value |
|--------|-------|
| Pages processed | 457 |
| Total rows | 24626 |
| Matched (all confidence levels) | 14295 (58.0%) |
| Within-page propagated | 91 |
| Cross-page propagated | 0 |
| Unmatched | 10331 |

## Match Confidence Breakdown

| Confidence | Count | % |
|------------|-------|---|
| no_match | 10331 | 42.0% |
| exact_sestiere_direct | 8080 | 32.8% |
| exact_sestiere_main_entrance_direct | 2678 | 10.9% |
| approximate_sestiere_first_variant_direct | 1239 | 5.0% |
| exact_sestiere_derived_from_parish | 664 | 2.7% |
| exact_sestiere_with_letter_direct | 557 | 2.3% |
| exact_sestiere_implied_from_landmark | 414 | 1.7% |
| exact_sestiere_main_entrance_derived_from_parish | 181 | 0.7% |
| approximate_sestiere_first_variant_derived_from_parish | 150 | 0.6% |
| exact_sestiere_main_entrance_implied_from_landmark | 137 | 0.6% |
| propagated_from_same

## Tables


In [ ]:

print("\n=== Section match rates ===")
section_display = pd.DataFrame(section_stats).T
display(section_display)

print("\n=== Sample matched rows ===")
display(df[df["match_confidence"].isin(MATCH_CONFIDENCES)][[
    "Name", "Address", "Location", "_page_num",
    "sestiere_parsed", "parish_parsed",
    "civic_num", "matched_sestiere", "matched_parish",
    "matched_lat", "matched_lon", "match_confidence"
]].head(20))

print("\n=== Unmatched rows with address info ===")
unmatched_with_info = df[
    ~df["match_confidence"].isin(MATCH_CONFIDENCES) &
    (df["civic_num"].notna() |
     df["sestiere_parsed"].notna() |
     df["parish_parsed"].notna())
][[
    "Name", "Address", "Location", "_page_num",
    "civic_num", "sestiere_parsed", "parish_parsed",
    "match_confidence", "match_note"
]]
display(unmatched_with_info.head(30))

# Save/upload helpers

def _get_run_folder_id():
    return get_run_folder_id(DRIVE_ROOT_FOLDER, DRIVE_GEOSPATIAL_FOLDER, DRIVE_OUTPUTS_FOLDER, RUN_NAME)





=== Section match rates ===


,total,matched,pct,no_num,not_in_ref,no_sestiere
cover_ads,7.0,5.0,71.4,2.0,0.0,2.0
government,2008.0,592.0,29.5,1351.0,65.0,938.0
indice_generale,9922.0,6268.0,63.2,2820.0,834.0,2593.0
industria,10092.0,6129.0,60.7,3246.0,717.0,2969.0
other,323.0,106.0,32.8,208.0,9.0,129.0
professionisti,1807.0,1162.0,64.3,548.0,97.0,504.0
religious,467.0,33.0,7.1,425.0,9.0,248.0



=== Sample matched rows ===


,Name,Address,Location,_page_num,sestiere_parsed,parish_parsed,civic_num,matched_sestiere,matched_parish,matched_lat,matched_lon,match_confidence
0,STTORE BORTOLI,Venezia - Cannaregio N. 5630,NaN,3,CANNAREGIO,NaN,5630.0,CANNAREGIO,San Canziano,45.439943,12.335942,exact_sestiere_main_entrance_direct
1,BOTTEGA DELL'ELETTRICITÀ,Cannaregio N. 5659,NaN,3,CANNAREGIO,NaN,5659.0,CANNAREGIO,San Canziano,45.439790,12.336705,exact_sestiere_direct
2,BOTTEGA DELLA LUCE,S. Luca N. 4511,NaN,3,NaN,San Luca,4511.0,SAN MARCO,San Luca,45.435549,12.335739,approximate_sestiere_first_variant_derived_fro...
4,CASA DEL REGALO,S. Luca N. 4480,NaN,3,NaN,San Luca,4480.0,SAN MARCO,San Luca,45.435720,12.335562,approximate_sestiere_first_variant_derived_fro...
6,EUGENIA SPERI,"Laboratorio : CANNAREGIO 1956, Negozi : S. MAR...",NaN,5,SAN MARCO,NaN,1956.0,SAN MARCO,Santa Maria Zobenigo,45.433910,12.333066,exact_sestiere_direct
111,olivetti FILIALE DI VENEZIA,San Marco - Bacino Orseolo 1224,NaN,41,SAN MARCO,NaN,1224.0,SAN MARCO,San Marco,45.434027,12.336859,exact_sestiere_main_entrance_direct
112,olivetti FILIALE DI VENEZIA,San Marco - Bacino Orseolo 1224,NaN,41,SAN MARCO,NaN,1224.0,SAN MARCO,San Marco,45.434027,12.336859,exact_sestiere_main_entrance_direct
194,Turci prof. Angiolo,Dorsoduro 1402.A.,NaN,44,DORSODURO,NaN,1402.0,DORSODURO,Santi Gervasio e Protasio,45.430294,12.324135,exact_sestiere_with_letter_direct
195,Favaretto Fisca ing. Giovanni,Castello 6449,NaN,44,CASTELLO,NaN,6449.0,CASTELLO,Santi Giovanni e Paolo,45.438375,12.344139,exact_sestiere_main_entrance_direct
196,Borin or. Iginio,Castello 3464,NaN,44,CASTELLO,NaN,3464.0,CASTELLO,San Giovanni Bragora,45.435494,12.345733,exact_sestiere_main_entrance_direct



=== Unmatched rows with address info ===


,Name,Address,Location,_page_num,civic_num,sestiere_parsed,parish_parsed,match_confidence,match_note
24,GIACOMO CARLO MC GUIGAN,Sta. Maria del Popolo,NaN,36,NaN,SAN POLO,NaN,no_match,no_civic_number
31,ALFREDO IDELFONSO SCHUSTER,Santi Silvestro e Martino ai Monti,NaN,36,NaN,NaN,San Silvestro,no_match,no_civic_number
32,EMANUELE CONCALVES CEREJEIRA,Santi Marcellino e Pietro,NaN,36,NaN,NaN,San Pietro,no_match,no_civic_number
33,LUIGI LAVITRANO,S. Silvestro in Capite,NaN,36,NaN,NaN,San Silvestro,no_match,no_civic_number
40,PIETRO FUMASONI BIONDI,S. Croce in Gerusalemme,NaN,37,NaN,SANTA CROCE,NaN,no_match,no_civic_number
41,MAURILIO FOSSATI,S. Croce in Gerusalemme,NaN,37,NaN,SANTA CROCE,NaN,no_match,no_civic_number
43,ELIA DALLA COSTA,S. Marco,NaN,37,NaN,SAN MARCO,NaN,no_match,no_civic_number
44,TEODORO INNITZER,S. Marco,NaN,37,NaN,SAN MARCO,NaN,no_match,no_civic_number
45,IGNAZIO GABRIELE TAPPONUI,S. Marco,NaN,37,NaN,SAN MARCO,NaN,no_match,no_civic_number
47,FRANCESCO SPELLMAN,"tit, SS. Giovanni e Paolo",NaN,37,NaN,NaN,Santi Giovanni e Paolo,no_match,no_civic_number


## Save outputs


In [10]:

df.columns = [str(c).strip() for c in df.columns]
df = df.loc[:, ~df.columns.duplicated(keep="first")]

if "page" in df.columns:
    df = df.drop(columns=["page"])
df = df.rename(columns={
    "_page_num":    "page_num",
    "_source_file": "source_file",
    "_section":     "section"
})

PRIORITY_COLS = [
    "Name", "Address", "Location", "Category", "Role", "Political Party",
    "Profession", "Additional Info", "Journal", "Notes",
    "page_num", "source_file", "section",
    "civic_num", "civic_lett", "sestiere_parsed", "parish_parsed",
    "sestiere_implied",
    "matched_lat", "matched_lon", "matched_parish", "matched_sestiere",
    "matched_lett", "all_doors", "match_confidence", "match_note",
    "geocoding_tier",
    "llm_decision", "llm_corrected_address", "llm_reasoning",
]
present_priority = [c for c in PRIORITY_COLS if c in df.columns]
remaining = sorted([c for c in df.columns if c not in PRIORITY_COLS
                    and not c.startswith("_")])
df = df[present_priority + remaining]

print("\nSaving outputs ...")
_svc, _run_folder_id = _get_run_folder_id()

# Geocoded CSV — matched rows only, no null coordinates
geocoded_df = df[df["matched_lat"].notna()].copy()
save_or_upload_csv(geocoded_df, OUT_CSV, _svc, _run_folder_id)
print(f"  Geocoded CSV    → {OUT_CSV.name} ({len(geocoded_df)} rows)")

# GeoJSON — matched rows only
if not geocoded_df.empty:
    gdf_cols = [c for c in geocoded_df.columns if c != "geocoding_tier"]
    gdf = gpd.GeoDataFrame(
        geocoded_df[gdf_cols],
        geometry=gpd.points_from_xy(
            geocoded_df["matched_lon"].astype(float),
            geocoded_df["matched_lat"].astype(float),
        ),
        crs="EPSG:4326",
    )
    save_or_upload_geojson(gdf, OUT_GEOJSON, _svc, _run_folder_id)
    print(f"  GeoJSON         → {OUT_GEOJSON.name} ({len(gdf)} points)")
else:
    print("  No matched rows — GeoJSON not created")

# Unmatched sestiere CSV — rows that failed the sestiere system
def extract_fail_reason(note):
    note = str(note or "")
    if "outside_venice" in note:
        return "outside_venice_scope"
    if "no_civic_number" in note:
        return "no_civic_number"
    if "not found in sestiere" in note:
        return "number_not_in_reference"
    if "no_sestiere_or_parish" in note:
        return "no_sestiere_or_parish"
    return "other"

GEOCODING_COLS_TO_DROP = [
    "matched_lat", "matched_lon", "matched_parish", "matched_sestiere",
    "matched_lett", "all_doors", "match_confidence", "geocoding_tier"
]

unmatched_sestiere = df[~df["match_confidence"].isin(MATCH_CONFIDENCES)].copy()
unmatched_sestiere["fail_reason"] = unmatched_sestiere["match_note"].apply(
    extract_fail_reason
)
unmatched_sestiere = unmatched_sestiere.drop(
    columns=[c for c in GEOCODING_COLS_TO_DROP if c in unmatched_sestiere.columns]
)
save_or_upload_csv(unmatched_sestiere, OUT_UNMATCHED_SESTIERE, _svc, _run_folder_id)
print(f"  Unmatched → {OUT_UNMATCHED_SESTIERE.name} ({len(unmatched_sestiere)} rows)")
print(f"  Fail reasons: {unmatched_sestiere['fail_reason'].value_counts().to_dict()}")

# Ambiguous CSV
ambiguous_df = df[df["all_doors"].notna()].copy()
save_or_upload_csv(ambiguous_df, OUT_AMBIGUOUS, _svc, _run_folder_id)
print(f"  Ambiguous CSV   → {OUT_AMBIGUOUS.name} ({len(ambiguous_df)} rows)")

# Markdown report
save_or_upload_text(report_text, OUT_REPORT, _svc, _run_folder_id)
print(f"  Report          → {OUT_REPORT.name}")

# Page-level stats CSV
page_stats_path = LOCAL_RUN_DIR / f"{FILE_TAG}_page_stats.csv"
save_or_upload_csv(page_stats_df, page_stats_path, _svc, _run_folder_id)
print(f"  Page stats CSV  → {page_stats_path.name}")

# Flagged review CSV
flagged_df = pd.concat([
    df[df["geocoding_tier"] == 4][[
        "Name", "Address", "Location", "page_num", "section",
        "civic_num", "sestiere_parsed", "parish_parsed",
        "matched_lat", "matched_lon", "matched_sestiere",
        "match_confidence", "geocoding_tier", "match_note"
    ]].assign(flag_reason="outside_venice_warning"),
    df[
        ~df["match_confidence"].isin(MATCH_CONFIDENCES) &
        df["match_note"].fillna("").str.contains("not found in sestiere", na=False)
    ][[
        "Name", "Address", "Location", "page_num", "section",
        "civic_num", "sestiere_parsed", "parish_parsed",
        "match_confidence", "match_note"
    ]].assign(
        matched_sestiere=None, geocoding_tier=0,
        flag_reason="number_not_in_reference"
    ),
], ignore_index=True)

flagged_path = LOCAL_RUN_DIR / f"{FILE_TAG}_flagged_review.csv"
save_or_upload_csv(flagged_df, flagged_path, _svc, _run_folder_id)
print(f"  Flagged review  → {flagged_path.name} ({len(flagged_df)} rows)")
print(f"  Flag breakdown  : {flagged_df['flag_reason'].value_counts().to_dict()}")

print(f"\n=== SECTION 12 DONE ===")
print(f"Run: {RUN_NAME}")
print(f"Matched (sestiere): {len(geocoded_df)} / {len(df)} ({100*len(geocoded_df)/len(df):.1f}%)")
print(f"Unmatched: {len(unmatched_sestiere)} rows")


Saving outputs ...
  Geocoded CSV    → all_pages_20260721_1901_geocoded.csv (14295 rows)
  GeoJSON         → all_pages_20260721_1901_geocoded.geojson (14295 points)
  Unmatched sestiere → all_pages_20260721_1901_unmatched_sestiere.csv (10331 rows)
  Fail reasons: {'no_civic_number': 8600, 'number_not_in_reference': 1731}
  Ambiguous CSV   → all_pages_20260721_1901_ambiguous.csv (4494 rows)
  Report          → all_pages_20260721_1901_report.md
  Page stats CSV  → all_pages_20260721_1901_page_stats.csv
  Flagged review  → all_pages_20260721_1901_flagged_review.csv (4377 rows)
  Flag breakdown  : {'outside_venice_warning': 2646, 'number_not_in_reference': 1731}

=== SECTION 12 DONE ===
Run: all_pages_20260721_1901
Matched (sestiere): 14295 / 24626 (58.0%)
Unmatched (sestiere): 10331 rows


## LLM repass on Tier-4 rows
Resolves outside-Venice ambiguity flags via an LLM call, cached per run tag.


In [ ]:
# Reads RUNTAG_llm_repass.csv if cached, else calls the API and saves decisions.
# keep_venice: kept in geocoded CSV. outside_venice: moved to unmatched_llm_repass CSV.

if RUN_LLM_REPASS:
    print("\n=== SECTION 12b — LLM REPASS ON TIER-4 ROWS ===")
    LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)

    tier4_idx = df[df["geocoding_tier"] == 4].index.tolist()
    print(f"  Tier-4 rows to process: {len(tier4_idx)}")

    # ── HELPER FUNCTIONS ─────────────────────────────────

    def read_clean_page_ocr(page_num):
        # reads page_X_ocr.txt, local first then Drive
        filename   = f"page_{page_num}_ocr.txt"
        local_path = CLEAN_PAGES / f"page_{page_num}" / filename
        if local_path.exists():
            return local_path.read_text(encoding="utf-8")
        index = build_clean_pages_index()
        if index:
            file_id = index.get(page_num, {}).get(filename)
            if file_id:
                return download_text(get_drive_service(), file_id)
        return None

    def llm_repass_batch(rows_batch, ocr_text, client):
        # sends flagged rows from one page to the LLM, keep_venice or outside_venice decisions
        rows_formatted = "\n".join([
            f"  Row {i+1}: Name={r['Name']} | "
            f"Address={r['Address']} | Location={r['Location']}"
            for i, r in enumerate(rows_batch)
        ])

        prompt = f"""You are analysing rows extracted from a 1947 Venetian commercial almanac.
The almanac uses the Venetian address system: sestiere name + civic number
(e.g. "Dorsoduro 3498-A", "Cannaregio 150", "S. Marco 4102").
The sestieri are: Dorsoduro, Cannaregio, San Marco, Santa Croce, San Polo, Castello, Giudecca. Sant'Elena is part of Castello, not its own sestiere.

ALL rows below come from the SAME almanac page. Use the full page context
to understand the structure — rows on the same page often share a header
address that gets propagated to sub-rows beneath it.

Some rows are flagged because their Location field contains a place outside
Venice (Mestre, Murano, Burano, Lido, Marghera, Zelarino etc.), even though
they may have a valid Venetian address in the Address field.

This outside-Venice name may be:
A) Secondary context — a workplace, branch office, or institution location
   that is NOT this row's primary address. The Venice address in the Address
   field is correct. Signs: "presso" (at/c/o), institution names, "filiale".
   → decision: keep_venice
B) The actual primary location — the Venice address in Address was wrongly
   propagated from a header row above. The OCR shows this row has no Venice
   address of its own.
   → decision: outside_venice

IMPORTANT:
- If Location contains "presso [place]" this is almost always case A — keep_venice.
- Only choose outside_venice if the OCR clearly shows this row belongs to
  an outside-Venice location with no Venice address of its own.
- If uncertain, always use keep_venice.

Here is the raw OCR text of this almanac page:
<ocr>
{ocr_text[:4000]}
</ocr>

Here are ALL flagged rows from this page to analyse:
{rows_formatted}

Respond ONLY with a valid JSON array, no preamble, no markdown fences.
Each element must have exactly these keys:
{{"row": <1-based int>, "decision": "keep_venice" or "outside_venice", "corrected_address": null, "reasoning": <one sentence>}}

corrected_address is always null."""

        response = client.chat.completions.create(
            model="gpt-4o",
            max_tokens=2000,
            messages=[{"role": "user", "content": prompt}]
        )

        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = re.sub(r"^```[a-z]*\n?", "", raw)
            raw = re.sub(r"\n?```$", "", raw)
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return [
                {"row": i+1, "decision": "keep_venice",
                 "corrected_address": None,
                 "reasoning": "JSON parse failed — defaulting to keep_venice"}
                for i in range(len(rows_batch))
            ]

    def apply_llm_decisions(df, decisions_df):
        # applies saved LLM decisions to df in-place, returns (df, kept, outside) counts
        kept = outside = 0

        df["llm_decision"]  = None
        df["llm_reasoning"] = None

        for _, res in decisions_df.iterrows():
            idx      = int(res["df_index"])
            decision = str(res["decision"])
            reason   = str(res["reasoning"])

            df.at[idx, "llm_decision"]  = decision
            df.at[idx, "llm_reasoning"] = reason

            if decision == "outside_venice":
                outside += 1
                df.at[idx, "matched_lat"]      = None
                df.at[idx, "matched_lon"]      = None
                df.at[idx, "matched_parish"]   = None
                df.at[idx, "matched_sestiere"] = None
                df.at[idx, "matched_lett"]     = None
                df.at[idx, "all_doors"]        = None
                df.at[idx, "match_confidence"] = "outside_venice_scope"
                df.at[idx, "geocoding_tier"]   = 0
                df.at[idx, "match_note"]       = (
                    f"LLM repass: outside_venice — {reason}"
                )
            else:
                # keep_venice (or any unrecognised decision) — no changes
                kept += 1
                df.at[idx, "match_note"] = (
                    df.at[idx, "match_note"] +
                    f" | LLM repass: keep_venice — {reason}"
                )

        return df, kept, outside

    # ── SKIP CONDITION ───────────────────────────────────

    if OUT_LLM_REPASS.exists():
        errors = 0
        print(f"  Decisions file found — loading from {OUT_LLM_REPASS.name}")
        print(f"  (no API calls — set RUN_LLM_REPASS=False to skip entirely)")
        decisions_df = pd.read_csv(OUT_LLM_REPASS)
        df, kept, outside = apply_llm_decisions(df, decisions_df)
        print(f"  Applied: keep_venice={kept} | outside_venice={outside}")

    else:
        print(f"  No decisions file — calling API for {len(tier4_idx)} rows ...")
        with open(Path(OPENAI_KEY_FILE)) as f:
            _key_params = dict(
                v.strip().split("=", 1) for v in f if "=" in v
            )
        llm_client       = openai.OpenAI(api_key=_key_params["api_key"])
        ocr_cache        = {}
        processed        = 0
        errors           = 0
        kept             = 0
        outside          = 0
        pages_processed  = 0
        decision_records = []

        tier4_df = df.loc[tier4_idx].copy()

        for page_num, page_group in tier4_df.groupby("page_num"):
            if page_num not in ocr_cache:
                ocr_cache[page_num] = read_clean_page_ocr(int(page_num)) or ""
            ocr_text = ocr_cache[page_num]

            page_indices = page_group.index.tolist()
            batch_rows = [
                {
                    "index":    idx,
                    "Name":     str(df.at[idx, "Name"]     or ""),
                    "Address":  str(df.at[idx, "Address"]  or ""),
                    "Location": str(df.at[idx, "Location"] or ""),
                }
                for idx in page_indices
            ]

            try:
                results = llm_repass_batch(batch_rows, ocr_text, llm_client)
                for res in results:
                    row_num = res.get("row", 1) - 1
                    if row_num >= len(page_indices):
                        continue
                    idx = page_indices[row_num]
                    decision_records.append({
                        "df_index":  idx,
                        "page_num":  page_num,
                        "Name":      str(df.at[idx, "Name"]     or ""),
                        "Address":   str(df.at[idx, "Address"]  or ""),
                        "Location":  str(df.at[idx, "Location"] or ""),
                        "decision":  res.get("decision", "keep_venice"),
                        "reasoning": res.get("reasoning", ""),
                    })
                processed       += len(page_indices)
                pages_processed += 1

            except Exception as e:
                print(f"    ERROR page {page_num}: {e} — retrying in 5s ...")
                time.sleep(5)
                try:
                    results = llm_repass_batch(batch_rows, ocr_text, llm_client)
                    for res in results:
                        row_num = res.get("row", 1) - 1
                        if row_num >= len(page_indices):
                            continue
                        idx = page_indices[row_num]
                        decision_records.append({
                            "df_index":  idx,
                            "page_num":  page_num,
                            "Name":      str(df.at[idx, "Name"]     or ""),
                            "Address":   str(df.at[idx, "Address"]  or ""),
                            "Location":  str(df.at[idx, "Location"] or ""),
                            "decision":  res.get("decision", "keep_venice"),
                            "reasoning": res.get("reasoning", ""),
                        })
                    processed       += len(page_indices)
                    pages_processed += 1
                except Exception as e2:
                    errors          += len(page_indices)
                    processed       += len(page_indices)
                    pages_processed += 1
                    for idx in page_indices:
                        decision_records.append({
                            "df_index":  idx,
                            "page_num":  page_num,
                            "Name":      str(df.at[idx, "Name"]     or ""),
                            "Address":   str(df.at[idx, "Address"]  or ""),
                            "Location":  str(df.at[idx, "Location"] or ""),
                            "decision":  "keep_venice",
                            "reasoning": f"API error after retry: {e2}",
                        })
                    print(f"    Retry also failed: {e2}")

            time.sleep(0.5)

            # Checkpoint every 10 pages
            if pages_processed % 10 == 0:
                _partial    = pd.DataFrame(decision_records)
                _dec_counts = _partial["decision"].value_counts().to_dict()
                print(f"\n  Progress: {processed}/{len(tier4_idx)} "
                      f"({100*processed/len(tier4_idx):.1f}%) | "
                      f"page={page_num} | pages_done={pages_processed} | errors={errors}")
                print(f"  Decisions so far: {_dec_counts}")
                _outside = _partial[_partial["decision"] == "outside_venice"]
                if len(_outside):
                    print(f"  outside_venice (most recent 3):")
                    print(_outside[["page_num","Address","Location","reasoning"]].tail(3).to_string(index=False))
                LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
                _partial.to_csv(OUT_LLM_REPASS, index=False)
                print(f"  Checkpoint saved → {OUT_LLM_REPASS.name}")

        # Final save
        decisions_df = pd.DataFrame(decision_records)
        decisions_df.to_csv(OUT_LLM_REPASS, index=False)
        print(f"\n  Decisions saved → {OUT_LLM_REPASS.name} ({len(decisions_df)} rows)")

        # Apply decisions
        df, kept, outside = apply_llm_decisions(df, decisions_df)

    # ── SUMMARY ──────────────────────────────────────────
    total_matched_after = df["matched_lat"].notna().sum()

    print(f"\n  === LLM REPASS SUMMARY ===")
    print(f"  keep_venice    : {kept}  ({100*kept/max(len(tier4_idx),1):.1f}%)")
    print(f"  outside_venice : {outside}  ({100*outside/max(len(tier4_idx),1):.1f}%)")
    print(f"  errors         : {errors}")

    print(f"\n  All outside_venice decisions ({len(decisions_df[decisions_df['decision']=='outside_venice'])} rows):")
    _ov = decisions_df[decisions_df["decision"] == "outside_venice"]
    if len(_ov):
        print(_ov[["page_num","Name","Address","Location","reasoning"]].to_string(index=False))
        print(f"  Pages affected: {sorted(_ov['page_num'].unique().tolist())}")
    else:
        print("  none")

    print(f"\n  Matched before repass : {total_matched}")
    print(f"  Matched after repass  : {total_matched_after}")
    print(f"  Net change            : {total_matched_after - total_matched:+d}")

    # ── RE-SAVE OUTPUTS AFTER REPASS ─────────────────────
    _svc, _run_folder_id = _get_run_folder_id()

    # Geocoded CSV — matched rows only, no null coordinates
    geocoded_after = df[df["matched_lat"].notna()].copy()
    save_or_upload_csv(geocoded_after, OUT_CSV, _svc, _run_folder_id)
    print(f"  Re-saved geocoded CSV → {OUT_CSV.name} ({len(geocoded_after)} rows)")

    # LLM repass unmatched — only rows removed by this step
    unmatched_llm = df[
        df["match_confidence"] == "outside_venice_scope"
    ].copy()
    unmatched_llm["fail_reason"] = unmatched_llm["match_note"].apply(
        lambda n: "outside_venice" if "outside_venice" in str(n) else "other"
    )
    unmatched_llm = unmatched_llm.drop(
        columns=[c for c in GEOCODING_COLS_TO_DROP if c in unmatched_llm.columns]
    )
    save_or_upload_csv(unmatched_llm, OUT_UNMATCHED_LLM, _svc, _run_folder_id)
    print(f"  Unmatched LLM repass → {OUT_UNMATCHED_LLM.name} ({len(unmatched_llm)} rows)")

    # Decisions log
    save_or_upload_csv(decisions_df, OUT_LLM_REPASS, _svc, _run_folder_id)
    print(f"  Saved decisions → {OUT_LLM_REPASS.name}")

    # GeoJSON
    matched_repass = df[df["matched_lat"].notna()].copy()
    gdf_cols = [c for c in matched_repass.columns if c != "geocoding_tier"]
    gdf_repass = gpd.GeoDataFrame(
        matched_repass[gdf_cols],
        geometry=gpd.points_from_xy(
            matched_repass["matched_lon"].astype(float),
            matched_repass["matched_lat"].astype(float),
        ),
        crs="EPSG:4326",
    )
    save_or_upload_geojson(gdf_repass, OUT_GEOJSON, _svc, _run_folder_id)
    print(f"  Re-saved GeoJSON → {OUT_GEOJSON.name} ({len(gdf_repass)} points)")

    print(f"\n=== SECTION 12b DONE ===")
    print(f"  Geocoded after repass  : {len(geocoded_after)} rows")
    print(f"  Removed by LLM repass  : {len(unmatched_llm)} rows")
    print(f"  Net change             : {total_matched_after - total_matched:+d}")

else:
    print("\n  Section 12b skipped (RUN_LLM_REPASS=False)")

print(f"\n=== DONE ===")
print(f"Run: {RUN_NAME}")
total_final = df["matched_lat"].notna().sum()
print(f"Total: {len(df)} rows | Matched: {total_final} ({100*total_final/max(len(df),1):.1f}%)")
if RUN_LLM_REPASS:
    print(f"  (after LLM repass — tier-4 rows cleaned)")


=== SECTION 12b — LLM REPASS ON TIER-4 ROWS ===
  Tier-4 rows to process: 4918
  No decisions file — calling API for 4918 rows ...

  Progress: 36/4918 (0.7%) | page=113 | pages_done=10 | errors=0
  Decisions so far: {'outside_venice': 24, 'keep_venice': 12}
  outside_venice (most recent 3):
 page_num      Address Location                                                                                       reasoning
      113 D. Duro 1335   Mestre The OCR shows this row belongs to an outside-Venice location with no Venice address of its own.
      113   Cann. 1825   Mestre The OCR shows this row belongs to an outside-Venice location with no Venice address of its own.
      113   Cann. 1249   Mestre The OCR shows this row belongs to an outside-Venice location with no Venice address of its own.
  Checkpoint saved → all_pages_20260618_2336_llm_repass.csv

  Progress: 142/4918 (2.9%) | page=123 | pages_done=20 | errors=0
  Decisions so far: {'outside_venice': 91, 'keep_venice': 51}
  out